In [ ]:
# ==============================================================================
# [1] 필수 패키지 설치 및 환경 설정
# ==============================================================================
!pip -q install catboost==1.2.8 category_encoders lightgbm

import os
import gc
import glob
import time
import joblib
import shutil
import zipfile
import subprocess
import sys
import torch
import numpy as np
import pandas as pd
from pathlib import Path

import lightgbm as lgb
from catboost import CatBoostClassifier
import category_encoders as ce
from scipy.stats import rankdata
from sklearn.model_selection import StratifiedKFold
from google.colab import files

# 경로 및 디렉터리 설정
if Path('/content/train.csv').exists():
    DATA_DIR = Path('/content')
elif Path('/content/data/train.csv').exists():
    DATA_DIR = Path('/content/data')
else:
    DATA_DIR = Path('.')

WORK_DIR = Path('/content/work')
MODEL_DIR = WORK_DIR / 'model'
ZIP_PATH = Path('/content/submit.zip')

MODEL_DIR.mkdir(parents=True, exist_ok=True)

ID_COL = "row_id"
TARGET_COL = "control_success"
TRAIN_PATH = DATA_DIR / 'train.csv'

assert TRAIN_PATH.exists(), f"❌ {TRAIN_PATH} 파일이 없습니다!"

print(f"1. 데이터 로드 중... ({TRAIN_PATH})")
df_train = pd.read_csv(TRAIN_PATH, encoding="utf-8-sig")

# 메모리 절약형 float32 캐스팅
for col in df_train.select_dtypes(include=['float64']).columns:
    df_train[col] = df_train[col].astype('float32')

target_enc_cols = ["top_bottom", "game_type", "base_state"]
for col in target_enc_cols:
    if col in df_train.columns:
        df_train[col] = df_train[col].astype("category")

features = [col for col in df_train.columns if col not in [ID_COL, TARGET_COL]]
X = df_train[features].copy()
y = df_train[TARGET_COL].values.astype(np.int8)

use_gpu = torch.cuda.is_available()

# ==============================================================================
# [2] LightGBM + CatBoost 5-Fold 학습 (float32 보장 & 실시간 로그)
# ==============================================================================
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_lgb = np.zeros(len(df_train), dtype=np.float32)
oof_cat = np.zeros(len(df_train), dtype=np.float32)

print(f"\n🚀 5-Fold 학습 시작... (하드웨어: {'GPU' if use_gpu else 'CPU'})")
total_t0 = time.time()

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    print(f"\n==================== Fold {fold + 1} / 5 시작 ====================")
    X_tr, y_tr = X.iloc[train_idx].copy(), y[train_idx]
    X_va, y_va = X.iloc[val_idx].copy(), y[val_idx]

    # Target Encoding 수행 후 float32 재캐스팅
    encoder = ce.TargetEncoder(cols=target_enc_cols, smoothing=10)
    X_tr_encoded = encoder.fit_transform(X_tr, y_tr)
    X_va_encoded = encoder.transform(X_va)

    for col in X_tr_encoded.columns:
        X_tr_encoded[col] = pd.to_numeric(X_tr_encoded[col], errors='coerce')
        X_va_encoded[col] = pd.to_numeric(X_va_encoded[col], errors='coerce')

    X_tr_encoded = X_tr_encoded.fillna(0.0).astype(np.float32)
    X_va_encoded = X_va_encoded.fillna(0.0).astype(np.float32)

    # 1) LightGBM
    model_lgb = lgb.LGBMClassifier(
        n_estimators=1000, learning_rate=0.08, max_depth=8, num_leaves=31,
        random_state=42, n_jobs=4
    )
    model_lgb.fit(
        X_tr_encoded, y_tr,
        eval_set=[(X_va_encoded, y_va)],
        callbacks=[
            lgb.early_stopping(30, verbose=False),
            lgb.log_evaluation(period=100)
        ]
    )
    oof_lgb[val_idx] = model_lgb.predict_proba(X_va_encoded)[:, 1].astype(np.float32)

    # 2) CatBoost
    model_cat = CatBoostClassifier(
        iterations=1000, learning_rate=0.08, depth=6, random_seed=42,
        verbose=100, thread_count=4, task_type="GPU" if use_gpu else "CPU"
    )
    model_cat.fit(
        X_tr_encoded, y_tr,
        eval_set=(X_va_encoded, y_va),
        early_stopping_rounds=30
    )
    oof_cat[val_idx] = model_cat.predict_proba(X_va_encoded)[:, 1].astype(np.float32)

    # 모델 파일 저장
    model_lgb.booster_.save_model(str(MODEL_DIR / f"model_lgb_fold{fold}.txt"))
    model_cat.save_model(str(MODEL_DIR / f"model_cat_fold{fold}.cbm"))
    joblib.dump(encoder, MODEL_DIR / f"encoder_fold{fold}.pkl")

    del X_tr, X_va, X_tr_encoded, X_va_encoded
    gc.collect()

print(f"\n🎉 전체 학습 완료 (총 소요 시간: {(time.time() - total_t0)/60:.1f}분)")

# ==============================================================================
# [3] Brier Skill Score (BSS) 평가 산식 100% 맞춤 무적 script.py 생성 및 검증
# ==============================================================================
SCRIPT_CONTENT = r'''import os
import glob
import joblib
import numpy as np
import pandas as pd
import lightgbm as lgb
from catboost import CatBoostClassifier
from pathlib import Path

ID_COL = "row_id"
TARGET_COL = "control_success"

def main():
    root = Path(__file__).resolve().parent
    test_path = root / "data" / "test.csv"
    sample_sub_path = root / "data" / "sample_submission.csv"
    out_path = root / "output" / "submission.csv"
    out_path.parent.mkdir(parents=True, exist_ok=True)

    if not test_path.exists():
        test_path = Path("./data/test.csv")
    if not sample_sub_path.exists():
        sample_sub_path = Path("./data/sample_submission.csv")

    df_test = pd.read_csv(test_path, encoding="utf-8-sig")
    df_sub = pd.read_csv(sample_sub_path, encoding="utf-8-sig")

    # 수치형/범주형 타입 정돈
    for col in df_test.select_dtypes(include=['float64']).columns:
        df_test[col] = df_test[col].astype('float32')

    cat_cols = ["top_bottom", "game_type", "base_state"]
    for col in cat_cols:
        if col in df_test.columns:
            df_test[col] = df_test[col].astype("category")

    drop_cols = [c for c in [ID_COL, TARGET_COL] if c in df_test.columns]
    X_test_raw = df_test.drop(columns=drop_cols).copy()

    model_dir = root / "model"
    lgb_files = sorted(glob.glob(str(model_dir / "model_lgb_fold*.txt")))
    cat_files = sorted(glob.glob(str(model_dir / "model_cat_fold*.cbm")))
    enc_files = sorted(glob.glob(str(model_dir / "encoder_fold*.pkl")))

    lgb_preds, cat_preds = [], []

    for lgb_f, cat_f, enc_f in zip(lgb_files, cat_files, enc_files):
        encoder = joblib.load(enc_f)
        X_encoded = encoder.transform(X_test_raw)

        for col in X_encoded.columns:
            X_encoded[col] = pd.to_numeric(X_encoded[col], errors='coerce')
        X_encoded = X_encoded.fillna(0.0).astype(np.float32)

        # LightGBM 확률 예측
        lgb_m = lgb.Booster(model_file=lgb_f)
        pred_lgb = lgb_m.predict(X_encoded)
        if pred_lgb.ndim > 1:
            pred_lgb = pred_lgb[:, 1]
        lgb_preds.append(pred_lgb)

        # CatBoost 확률 예측
        cb_m = CatBoostClassifier()
        cb_m.load_model(cat_f)
        pred_cat = cb_m.predict_proba(X_encoded)[:, 1]
        cat_preds.append(pred_cat)

    # Brier Score 보존을 위한 원본 확률값 5:5 블렌딩 & 클리핑
    mean_lgb = np.mean(lgb_preds, axis=0)
    mean_cat = np.mean(cat_preds, axis=0)
    final_preds = np.clip(0.5 * mean_lgb + 0.5 * mean_cat, 0.0, 1.0)

    # row_id 안전 1:1 매핑
    pred_map = dict(zip(df_test[ID_COL].astype(str), final_preds))
    df_sub[TARGET_COL] = df_sub[ID_COL].astype(str).map(pred_map).fillna(0.5)

    df_sub.to_csv(out_path, index=False, encoding="utf-8-sig")
    print("✅ Inference Finished Successfully!")

if __name__ == "__main__":
    main()
'''

(WORK_DIR / 'script.py').write_text(SCRIPT_CONTENT, encoding='utf-8')
(WORK_DIR / 'requirements.txt').write_text("catboost==1.2.8\nlightgbm\ncategory_encoders\n", encoding='utf-8')

# ==============================================================================
# [4] 가상 테스트 구동 및 submit.zip 압축/다운로드
# ==============================================================================
print("\n3. script.py 사전 가상 테스트 진행 중...")
package_data = WORK_DIR / 'data'
package_output = WORK_DIR / 'output'
package_data.mkdir(exist_ok=True); package_output.mkdir(exist_ok=True)

train_sample = df_train.head(5).copy()
train_sample.drop(columns=[TARGET_COL], errors='ignore').to_csv(package_data / 'test.csv', index=False, encoding='utf-8-sig')
train_sample[[ID_COL, TARGET_COL]].to_csv(package_data / 'sample_submission.csv', index=False, encoding='utf-8-sig')

result = subprocess.run([sys.executable, str(WORK_DIR / 'script.py')], cwd=WORK_DIR, capture_output=True, text=True)

if result.returncode != 0:
    print("❌ script.py 사전 테스트 실행 오류!")
    print(result.stderr)
    raise RuntimeError("검증 실패")
else:
    print("✅ script.py 사전 테스트 성공!")

shutil.rmtree(package_data)
shutil.rmtree(package_output)

# submit.zip 패킹 및 자동 다운로드
if ZIP_PATH.exists():
    ZIP_PATH.unlink()

with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as zipf:
    zipf.write(WORK_DIR / 'script.py', arcname='script.py')
    zipf.write(WORK_DIR / 'requirements.txt', arcname='requirements.txt')

    for file in MODEL_DIR.rglob('*'):
        if file.is_file():
            arcname = file.relative_to(WORK_DIR)
            zipf.write(file, arcname=arcname)

print("\n🎉 Brier Score 산식 맞춤 submit.zip 작성이 완료되었습니다!")
files.download(ZIP_PATH)

1. 데이터 로드 중... (/content/train.csv)

🚀 5-Fold 학습 시작... (진행 상황 실시간 출력 ON)

==================== Fold 1 / 5 시작 ====================
[Fold 1] Target Encoding 변환 중...
[Fold 1] LightGBM 학습 중...
[100]	valid_0's binary_logloss: 0.683007
[200]	valid_0's binary_logloss: 0.682593
[300]	valid_0's binary_logloss: 0.68234
[400]	valid_0's binary_logloss: 0.682145
[500]	valid_0's binary_logloss: 0.682007
[600]	valid_0's binary_logloss: 0.681933
[700]	valid_0's binary_logloss: 0.681891
[800]	valid_0's binary_logloss: 0.681876
[Fold 1] CatBoost 학습 중...
0:	learn: 0.6919929	test: 0.6919569	best: 0.6919569 (0)	total: 394ms	remaining: 6m 33s
100:	learn: 0.6828478	test: 0.6834982	best: 0.6834982 (100)	total: 41.7s	remaining: 6m 10s
200:	learn: 0.6818346	test: 0.6830350	best: 0.6830350 (200)	total: 1m 22s	remaining: 5m 26s
300:	learn: 0.6808091	test: 0.6826892	best: 0.6826892 (300)	total: 2m 7s	remaining: 4m 55s
400:	learn: 0.6798879	test: 0.6824798	best: 0.6824721 (394)	total: 2m 48s	remaining: 4m 11s
500:	

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>